# Curso de Auditoría de Modelos – CaixaBank  
## Día 2

> Estos notebooks parten de un **CSV existente** con una columna objetivo llamada `default`.
> No generan datos sintéticos.

## Filosofía docente
- todo paso a paso
- sin esconder la lógica principal dentro de funciones
- con explicación conceptual entre bloques
- con ejercicios y solucionario

# Notebook 2 · XGBoost, AdaBoost y CatBoost

Comparativa de boosting paso a paso.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay
)

csv_path = "dataset_bancario_sintetico.csv"
df = pd.read_csv(csv_path)

print("Dimensiones:", df.shape)
display(df.head())
display(df.info())

print("\nDistribución de default:")
display(df["default"].value_counts(dropna=False))
display(df["default"].value_counts(normalize=True).rename("proporcion"))

In [ ]:
columnas_a_excluir = ["default", "cliente_id"]
columnas_presentes_a_excluir = [c for c in columnas_a_excluir if c in df.columns]

X = df.drop(columns=columnas_presentes_a_excluir)
y = df["default"]

print("Dimensiones de X:", X.shape)
print("Dimensiones de y:", y.shape)

variables_numericas = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
variables_categoricas = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("\nVariables numéricas:")
print(variables_numericas)
print("\nVariables categóricas:")
print(variables_categoricas)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("\nTamaño train:", X_train.shape)
print("Tamaño test :", X_test.shape)

In [ ]:
transformador_numerico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

transformador_categorico = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", transformador_numerico, variables_numericas),
    ("cat", transformador_categorico, variables_categoricas)
])

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

modelo_adaboost = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=200,
        learning_rate=0.05,
        random_state=42
    ))
])

modelo_adaboost.fit(X_train, y_train)
pred_adaboost = modelo_adaboost.predict(X_test)
proba_adaboost = modelo_adaboost.predict_proba(X_test)[:, 1]
print("AdaBoost AUC:", round(roc_auc_score(y_test, proba_adaboost), 4))